In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    print("Not running in Google Colab; skipping Drive mount.")


In [ ]:
from pathlib import Path
import os


def resolve_data_path(filename: str) -> str:
    """
    Resolve data files in local Jupyter or Google Colab.
    In Colab, mount Google Drive first and place data under:
    /content/drive/MyDrive/feature-selection-experiments/data
    """
    candidates = []

    env_data_dir = os.environ.get("FEATURE_SELECTION_DATA_DIR")
    if env_data_dir:
        candidates.append(Path(env_data_dir) / filename)

    cwd = Path.cwd()
    candidates.extend([
        cwd / "data" / filename,
        cwd.parent / "data" / filename,
        Path("/content/data") / filename,
        Path("/content/feature-selection-experiments/data") / filename,
        Path("/content/drive/MyDrive/feature-selection-experiments/data") / filename,
        Path("/content/drive/MyDrive/data") / filename,
        Path(r"D:\feature-selection-experiments\data") / filename,
    ])

    for path in candidates:
        if path.exists():
            return str(path)

    search_roots = [
        Path("/content"),
        Path("/content/drive/MyDrive"),
    ]
    for root in search_roots:
        if not root.exists():
            continue
        matches = list(root.glob(f"**/{filename}"))
        for match in matches:
            if match.parent.name == "data":
                return str(match)
        if matches:
            return str(matches[0])

    checked = "\n".join(str(path) for path in candidates)
    raise FileNotFoundError(
        f"Could not find {filename}. Checked:\n{checked}\n\n"
        "Upload the data folder to /content/data, or mount Google Drive and place it at "
        "/content/drive/MyDrive/feature-selection-experiments/data."
    )


DATA_DIR = Path(resolve_data_path("X_train.csv")).parent
if Path("/content/drive/MyDrive").exists():
    RESULT_DIR = Path("/content/drive/MyDrive/feature-selection-experiments/results/deql-spark-diversity")
else:
    RESULT_DIR = Path.cwd() / "results" / "deql-spark-diversity"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = RESULT_DIR


def output_path(filename: str) -> str:
    return str(RESULT_DIR / filename)


print(f"Using DATA_DIR: {DATA_DIR}")
print(f"Using RESULT_DIR: {RESULT_DIR}")


In [25]:
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Tuple, List, Dict, Optional

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.svm import LinearSVC
import time


In [26]:
@dataclass
class QLDEConfig:
    population_size: int = 20
    max_generations: int = 30
    crossover_rate: float = 0.9
    init_F_min: float = 0.3
    init_F_max: float = 0.8
    q_learning_rate: float = 0.1   # alpha
    q_discount: float = 0.9        # gamma
    softmax_temperature: float = 1.0
    random_state: int = 42
    cv_splits: int = 5

    # Fitness = alpha_metric * macro_f1 - beta_size * feature_ratio
    alpha_metric: float = 0.95
    beta_size: float = 0.05

    # Các action điều chỉnh F
    actions: Tuple[float, ...] = (-0.05, 0.0, 0.05)

    # Ngưỡng biến continuous -> binary mask
    threshold: float = 0.5

    # Giới hạn F
    F_lower_bound: float = 0.1
    F_upper_bound: float = 1.0

    # Nếu cá thể chọn ít hơn min_features thì tự sửa
    min_features: int = 1

    # Dùng top ratio để cluster-guided mutation như paper
    top_ratio: float = 0.5
    val_size: float = 0.2

    archive_size: int = 15
    archive_min_hamming: float = 0.10       # khoảng cách tối thiểu để xem là "khác"
    archive_novelty_k: int = 3              # dùng k láng giềng gần nhất để tính novelty
    leader_novelty_weight: float = 0.7
    leader_fitness_weight: float = 0.3

In [28]:
def load_data(
    x_path: str,
    y_path: str
) -> Tuple[pd.DataFrame, np.ndarray]:
    X = pd.read_csv(x_path)
    y_df = pd.read_csv(y_path)

    if y_df.shape[1] != 1:
        raise ValueError("y_train.csv phải chỉ có đúng 1 cột nhãn.")

    y = y_df.iloc[:, 0].values

    # Encode nhãn nếu là chuỗi
    if y.dtype == object or str(y.dtype).startswith("string"):
        le = LabelEncoder()
        y = le.fit_transform(y)

    return X, y

In [29]:
def logistic_map_init(
    population_size: int,
    n_features: int,
    rng: np.random.Generator,
    mu: float = 3.99
) -> np.ndarray:
    """
    Khởi tạo quần thể bằng Logistic chaotic mapping.
    """
    pop = np.zeros((population_size, n_features), dtype=float)

    for i in range(population_size):
        phi = rng.uniform(0.001, 0.999)
        for j in range(n_features):
            phi = mu * phi * (1.0 - phi)
            pop[i, j] = phi

    return pop


def continuous_to_mask(
    vector: np.ndarray,
    threshold: float,
    min_features: int,
    rng: np.random.Generator
) -> np.ndarray:
    """
    Convert continuous vector [0,1] -> binary mask.
    """
    mask = (vector > threshold).astype(int)

    if mask.sum() < min_features:
        # ép chọn ngẫu nhiên min_features cột
        idx = rng.choice(len(mask), size=min_features, replace=False)
        mask[idx] = 1

    return mask


def select_action_softmax(
    q_table: np.ndarray,
    state: int,
    temperature: float,
    rng: np.random.Generator
) -> int:
    """
    Chọn action theo softmax trên hàng Q[state].
    state: 0 hoặc 1
    """
    q_values = q_table[state].copy()

    if temperature <= 0:
        return int(np.argmax(q_values))

    q_scaled = q_values / temperature
    q_scaled = q_scaled - np.max(q_scaled)
    probs = np.exp(q_scaled)
    probs = probs / probs.sum()

    return int(rng.choice(len(q_values), p=probs))


def evaluate_feature_subset(
    X: pd.DataFrame,
    y: np.ndarray,
    mask: np.ndarray,
    config: QLDEConfig
) -> Tuple[float, float, int]:
    """
    Evaluate một tập feature bằng 1 lần chia train/validation.
    Trả về:
    - fitness
    - macro_f1 trên validation
    - số lượng feature được chọn
    """
    selected_cols = X.columns[mask == 1]
    X_sel = X[selected_cols]

    if X_sel.shape[1] == 0:
        return -1e9, 0.0, 0

    X_train_fs, X_val_fs, y_train_fs, y_val_fs = train_test_split(
        X_sel,
        y,
        test_size=config.val_size,
        random_state=config.random_state,
        stratify=y
    )

    clf = LinearSVC(
    C=1.0,
    max_iter=2000,
    random_state=config.random_state)

    clf.fit(X_train_fs, y_train_fs)
    y_pred = clf.predict(X_val_fs)

    macro_f1 = f1_score(y_val_fs, y_pred, average="macro")
    feature_ratio = X_sel.shape[1] / X.shape[1]

    fitness = (
        config.alpha_metric * macro_f1
        - config.beta_size * feature_ratio
    )

    return fitness, macro_f1, X_sel.shape[1]

def hamming_distance_ratio(mask_a: np.ndarray, mask_b: np.ndarray) -> float:
    """
    Tỷ lệ bit khác nhau giữa 2 mask.
    """
    if len(mask_a) != len(mask_b):
        raise ValueError("Hai mask phải cùng kích thước.")
    return float(np.mean(mask_a != mask_b))



def hamming_distance_ratio(mask_a: np.ndarray, mask_b: np.ndarray) -> float:
    """Return the normalized Hamming distance between two binary masks."""
    mask_a = np.asarray(mask_a).astype(int)
    mask_b = np.asarray(mask_b).astype(int)
    if mask_a.shape != mask_b.shape:
        raise ValueError("Two masks must have the same shape.")
    return float(np.mean(mask_a != mask_b))


def continuous_population_to_masks(
    population: np.ndarray,
    threshold: float = 0.5,
    min_features: int = 1,
    use_sigmoid: bool = False
) -> np.ndarray:
    """Convert a continuous population to binary feature masks for metric tracking."""
    values = 1.0 / (1.0 + np.exp(-population)) if use_sigmoid else np.asarray(population)
    masks = (values > threshold).astype(int)

    if min_features > 0:
        for i in range(masks.shape[0]):
            if masks[i].sum() < min_features:
                top_idx = np.argsort(values[i])[-min_features:]
                masks[i, top_idx] = 1

    return masks


def compute_population_diversity(population_masks: np.ndarray) -> float:
    """Average pairwise Hamming distance among all masks in the current population."""
    population_masks = np.asarray(population_masks).astype(int)
    n_individuals = population_masks.shape[0]
    if n_individuals < 2:
        return 0.0

    total_distance = 0.0
    pair_count = 0
    for i in range(n_individuals):
        for j in range(i + 1, n_individuals):
            total_distance += hamming_distance_ratio(population_masks[i], population_masks[j])
            pair_count += 1

    return float(total_distance / pair_count)


def compute_exploration_exploitation(
    new_masks: np.ndarray,
    previous_masks: np.ndarray,
    similarity_threshold: float = 0.1
) -> Tuple[float, float]:
    """
    Exploration is the fraction of new masks whose nearest previous neighbor
    is farther than the threshold. Exploitation is the complementary fraction.
    """
    new_masks = np.asarray(new_masks).astype(int)
    previous_masks = np.asarray(previous_masks).astype(int)
    if len(new_masks) == 0 or len(previous_masks) == 0:
        return 0.0, 0.0

    exploration_count = 0
    for new_mask in new_masks:
        nearest_distance = min(
            hamming_distance_ratio(new_mask, previous_mask)
            for previous_mask in previous_masks
        )
        if nearest_distance > similarity_threshold:
            exploration_count += 1

    exploration = exploration_count / len(new_masks)
    exploitation = 1.0 - exploration
    return float(exploration), float(exploitation)

def compute_archive_novelty(
    archive: List[Dict],
    k: int = 3
) -> np.ndarray:
    """
    Tính novelty cho từng nghiệm trong archive dựa trên Hamming distance.
    Novelty cao = nghiệm nằm ở vùng thưa hơn.
    """
    n = len(archive)
    if n == 0:
        return np.array([])
    if n == 1:
        return np.array([1.0], dtype=float)

    dist_mat = np.zeros((n, n), dtype=float)

    for i in range(n):
        for j in range(i + 1, n):
            d = hamming_distance_ratio(archive[i]["mask"], archive[j]["mask"])
            dist_mat[i, j] = d
            dist_mat[j, i] = d

    novelty = np.zeros(n, dtype=float)
    for i in range(n):
        dists = np.sort(dist_mat[i][np.arange(n) != i])
        kk = min(k, len(dists))
        novelty[i] = float(np.mean(dists[:kk])) if kk > 0 else 1.0

    return novelty


def try_add_to_archive(
    archive: List[Dict],
    candidate: Dict,
    max_size: int,
    min_hamming: float,
    novelty_k: int
) -> List[Dict]:
    """
    Thêm candidate vào archive theo kiểu diversity-aware:
    - Nếu quá giống 1 nghiệm sẵn có:
        + candidate tốt hơn -> thay thế
        + candidate kém hơn -> bỏ
    - Nếu còn chỗ -> thêm
    - Nếu đầy -> thêm tạm rồi loại nghiệm vừa kém fitness vừa kém novelty
    """
    if len(archive) == 0:
        archive.append(candidate)
        return archive

    # 1) kiểm tra nghiệm quá giống
    best_similar_idx = None
    best_similar_dist = float("inf")

    for idx, item in enumerate(archive):
        d = hamming_distance_ratio(candidate["mask"], item["mask"])
        if d < min_hamming and d < best_similar_dist:
            best_similar_dist = d
            best_similar_idx = idx

    if best_similar_idx is not None:
        if candidate["fitness"] > archive[best_similar_idx]["fitness"]:
            archive[best_similar_idx] = candidate
        return archive

    # 2) còn chỗ thì thêm
    if len(archive) < max_size:
        archive.append(candidate)
        return archive

    # 3) archive đầy -> thêm tạm rồi loại 1 nghiệm dư thừa
    temp_archive = archive + [candidate]
    novelty = compute_archive_novelty(temp_archive, k=novelty_k)
    fitness = np.array([item["fitness"] for item in temp_archive], dtype=float)

    # normalize fitness về [0,1]
    fit_min, fit_max = float(np.min(fitness)), float(np.max(fitness))
    if fit_max - fit_min < 1e-12:
        fit_norm = np.ones_like(fitness)
    else:
        fit_norm = (fitness - fit_min) / (fit_max - fit_min)

    # điểm thấp = vừa ít mới lạ vừa fitness yếu -> loại
    removal_score = 0.5 * novelty + 0.5 * fit_norm
    remove_idx = int(np.argmin(removal_score))

    archive = [item for i, item in enumerate(temp_archive) if i != remove_idx]
    return archive


def select_leader_from_archive(
    archive: List[Dict],
    rng: np.random.Generator,
    novelty_k: int,
    novelty_weight: float,
    fitness_weight: float
) -> Dict:
    """
    Chọn leader từ archive.
    Tinh thần MOHEOA:
    - ưu tiên nghiệm ở vùng thưa (novelty cao)
    - nhưng vẫn cân nhắc fitness
    """
    if len(archive) == 0:
        raise ValueError("Archive rỗng, không thể chọn leader.")

    if len(archive) == 1:
        return archive[0]

    novelty = compute_archive_novelty(archive, k=novelty_k)
    fitness = np.array([item["fitness"] for item in archive], dtype=float)

    # normalize
    nov_min, nov_max = float(np.min(novelty)), float(np.max(novelty))
    if nov_max - nov_min < 1e-12:
        novelty_norm = np.ones_like(novelty)
    else:
        novelty_norm = (novelty - nov_min) / (nov_max - nov_min)

    fit_min, fit_max = float(np.min(fitness)), float(np.max(fitness))
    if fit_max - fit_min < 1e-12:
        fitness_norm = np.ones_like(fitness)
    else:
        fitness_norm = (fitness - fit_min) / (fit_max - fit_min)

    scores = novelty_weight * novelty_norm + fitness_weight * fitness_norm
    scores = np.clip(scores, 1e-12, None)
    probs = scores / scores.sum()

    idx = int(rng.choice(len(archive), p=probs))
    return archive[idx]
    
class QLDEFeatureSelector:
    def __init__(self, config: Optional[QLDEConfig] = None):
        self.config = config if config is not None else QLDEConfig()
        self.rng = np.random.default_rng(self.config.random_state)

        self.best_mask_: Optional[np.ndarray] = None
        self.best_fitness_: float = -np.inf
        self.best_macro_f1_: float = 0.0
        self.best_num_features_: int = 0
        self.history_: List[Dict] = []
        self.optimization_metrics_: Optional[pd.DataFrame] = None
        self.q_table_: Optional[np.ndarray] = None
        self.feature_selection_time_: float = 0.0

        self.archive_: List[Dict] = []
        self.population_fitness_history_ = []
        self.population_state_history_ = []

    def _initialize_population(self, n_features: int) -> Tuple[np.ndarray, np.ndarray]:
        population = logistic_map_init(
            population_size=self.config.population_size,
            n_features=n_features,
            rng=self.rng
        )

        # F riêng cho từng cá thể như paper
        F_values = self.rng.uniform(
            self.config.init_F_min,
            self.config.init_F_max,
            size=self.config.population_size
        )

        return population, F_values

    def _make_archive_item(
        self,
        vector: np.ndarray,
        mask: np.ndarray,
        fitness: float,
        macro_f1: float,
        num_features: int
    ) -> Dict:
        return {
            "vector": vector.copy(),
            "mask": mask.copy(),
            "fitness": float(fitness),
            "macro_f1": float(macro_f1),
            "num_features": int(num_features),
        }

    def _update_archive(
        self,
        vector: np.ndarray,
        mask: np.ndarray,
        fitness: float,
        macro_f1: float,
        num_features: int
    ) -> None:
        candidate = self._make_archive_item(
            vector=vector,
            mask=mask,
            fitness=fitness,
            macro_f1=macro_f1,
            num_features=num_features
        )

        self.archive_ = try_add_to_archive(
            archive=self.archive_,
            candidate=candidate,
            max_size=self.config.archive_size,
            min_hamming=self.config.archive_min_hamming,
            novelty_k=self.config.archive_novelty_k
        )
    

    def fit(self, X: pd.DataFrame, y: np.ndarray):
        import time
    
        fs_start = time.perf_counter()
    
        if not isinstance(X, pd.DataFrame):
            X_df_full = pd.DataFrame(X)
        else:
            X_df_full = X.copy()
        y = np.asarray(y)
    
        n_features = X_df_full.shape[1]
    
        # Q-table: 2 states x num_actions
        q_table = np.zeros((2, len(self.config.actions)), dtype=float)
    
        population, F_values = self._initialize_population(n_features)
    
        fitnesses = np.zeros(self.config.population_size)
        macro_f1s = np.zeros(self.config.population_size)
        num_features = np.zeros(self.config.population_size, dtype=int)
    
        # reset archive mỗi lần fit
        self.archive_ = []
    
        # =========================
        # 1) ĐÁNH GIÁ QUẦN THỂ BAN ĐẦU
        # =========================
        init_eval_start = time.perf_counter()
    
        for i in range(self.config.population_size):
            mask = continuous_to_mask(
                population[i],
                self.config.threshold,
                self.config.min_features,
                self.rng
            )
    
            fit_i, f1_i, nf_i = evaluate_feature_subset(
                X_df_full, y, mask, self.config
            )
    
            fitnesses[i] = fit_i
            macro_f1s[i] = f1_i
            num_features[i] = nf_i
    
            # update archive
            self._update_archive(
                vector=population[i],
                mask=mask,
                fitness=fit_i,
                macro_f1=f1_i,
                num_features=nf_i
            )
    
            if fit_i > self.best_fitness_:
                self.best_fitness_ = fit_i
                self.best_mask_ = mask.copy()
                self.best_macro_f1_ = f1_i
                self.best_num_features_ = nf_i
    
        init_eval_time = time.perf_counter() - init_eval_start
        print(f"[Timing] Initial population evaluation: {init_eval_time:.4f} seconds")
    
        # =========================
        # 2) VÒNG LẶP CHÍNH
        # =========================
        for gen in range(self.config.max_generations):
            gen_start = time.perf_counter()
    
            previous_masks_for_metrics = continuous_population_to_masks(
                population,
                threshold=self.config.threshold,
                min_features=self.config.min_features
            )
            trial_masks_for_metrics = np.zeros_like(previous_masks_for_metrics)

            new_population = population.copy()
            new_fitnesses = fitnesses.copy()
            new_macro_f1s = macro_f1s.copy()
            new_num_features = num_features.copy()

            for i in range(len(population)):
                mask_i = continuous_to_mask(
                    population[i],
                    self.config.threshold,
                    self.config.min_features,
                    self.rng
                )
            
                self.population_state_history_.append({
                    "generation": gen + 1,
                    "individual": i,
                    "fitness": float(fitnesses[i]),
                    "num_features": int(num_features[i]),
                    "mask": mask_i.tolist()
                })
    
            sorted_idx = np.argsort(-fitnesses)
            top_k = max(1, int(self.config.top_ratio * self.config.population_size))
            elite_indices = sorted_idx[:top_k]
    
            gen_improvements = 0
            eval_start = time.perf_counter()
    
            for i in range(self.config.population_size):
                parent = population[i].copy()
                parent_fitness = fitnesses[i]
    
                current_state = 1
    
                action_idx = select_action_softmax(
                    q_table=q_table,
                    state=current_state,
                    temperature=self.config.softmax_temperature,
                    rng=self.rng
                )
                delta_F = self.config.actions[action_idx]
    
                F_i = F_values[i] + delta_F
                F_i = float(np.clip(
                    F_i,
                    self.config.F_lower_bound,
                    self.config.F_upper_bound
                ))
    
                # Chọn leader từ archive thay vì elite ngẫu nhiên
                leader_item = select_leader_from_archive(
                    archive=self.archive_,
                    rng=self.rng,
                    novelty_k=self.config.archive_novelty_k,
                    novelty_weight=self.config.leader_novelty_weight,
                    fitness_weight=self.config.leader_fitness_weight
                )
                leader_vec = leader_item["vector"]
    
                candidates = [idx for idx in range(self.config.population_size) if idx != i]
                r1, r2 = self.rng.choice(candidates, size=2, replace=False)
    
                mutant = leader_vec + F_i * (population[r1] - population[r2])
                mutant = np.clip(mutant, 0.0, 1.0)
    
                trial = parent.copy()
                j_rand = self.rng.integers(0, n_features)
    
                for j in range(n_features):
                    if self.rng.random() <= self.config.crossover_rate or j == j_rand:
                        trial[j] = mutant[j]
    
                trial = np.clip(trial, 0.0, 1.0)
    
                trial_mask = continuous_to_mask(
                    trial,
                    self.config.threshold,
                    self.config.min_features,
                    self.rng
                )
    
                trial_fitness, trial_f1, trial_nf = evaluate_feature_subset(
                    X_df_full, y, trial_mask, self.config
                )
                trial_masks_for_metrics[i] = trial_mask
    
                if trial_fitness > parent_fitness:
                    reward = 1.0
                    next_state = 0
                else:
                    reward = 0.0
                    next_state = 1
    
                best_next_q = np.max(q_table[next_state])
                old_q = q_table[current_state, action_idx]
                q_table[current_state, action_idx] = old_q + self.config.q_learning_rate * (
                    reward + self.config.q_discount * best_next_q - old_q
                )
    
                if trial_fitness > parent_fitness:
                    new_population[i] = trial
                    new_fitnesses[i] = trial_fitness
                    new_macro_f1s[i] = trial_f1
                    new_num_features[i] = trial_nf
                    F_values[i] = F_i
                    gen_improvements += 1
    
                    # update archive
                    self._update_archive(
                        vector=trial,
                        mask=trial_mask,
                        fitness=trial_fitness,
                        macro_f1=trial_f1,
                        num_features=trial_nf
                    )
    
                    if trial_fitness > self.best_fitness_:
                        self.best_fitness_ = trial_fitness
                        self.best_mask_ = trial_mask.copy()
                        self.best_macro_f1_ = trial_f1
                        self.best_num_features_ = trial_nf
    
            eval_time = time.perf_counter() - eval_start
    
            population = new_population
            fitnesses = new_fitnesses
            macro_f1s = new_macro_f1s
            num_features = new_num_features

            current_masks_for_metrics = continuous_population_to_masks(
                population,
                threshold=self.config.threshold,
                min_features=self.config.min_features
            )
            diversity = compute_population_diversity(current_masks_for_metrics)
            exploration_threshold = getattr(self.config, "exploration_threshold", 0.1)
            exploration, exploitation = compute_exploration_exploitation(
                new_masks=trial_masks_for_metrics,
                previous_masks=previous_masks_for_metrics,
                similarity_threshold=exploration_threshold
            )

            for i in range(len(fitnesses)):
                self.population_fitness_history_.append({
                    "generation": gen + 1,
                    "individual": i,
                    "fitness": float(fitnesses[i])
                })
    
            self.history_.append({
                "generation": gen + 1,
                "best_fitness": float(np.max(fitnesses)),
                "mean_fitness": float(np.mean(fitnesses)),
                "best_macro_f1": float(np.max(macro_f1s)),
                "mean_macro_f1": float(np.mean(macro_f1s)),
                "best_num_features": int(num_features[np.argmax(fitnesses)]),
                "improvements": gen_improvements,
                "diversity": diversity,
                "exploration": exploration,
                "exploitation": exploitation,
                "exploration_threshold": exploration_threshold,
                "archive_size": len(self.archive_),
                "eval_time_sec": eval_time
            })
    
            gen_time = time.perf_counter() - gen_start
    
            print(
                f"Generation {gen+1:02d} | "
                f"Best fitness = {np.max(fitnesses):.5f} | "
                f"Best macro-F1 = {np.max(macro_f1s):.5f} | "
                f"Selected features = {num_features[np.argmax(fitnesses)]} | "
                f"Improvements = {gen_improvements} | "
                f"Diversity = {diversity:.4f} | "
                f"Exploration = {exploration:.4f} | "
                f"Exploitation = {exploitation:.4f} | "
                f"Archive = {len(self.archive_)} | "
                f"Eval time = {eval_time:.4f}s | "
                f"Total gen time = {gen_time:.4f}s"
            )
    
        self.optimization_metrics_ = pd.DataFrame(self.history_)
        self.optimization_metrics_.to_csv(output_path("optimization_metrics_deql_diversity_with_formulas.csv"), index=False)
        print("[Metrics] Saved optimization_metrics_deql_diversity_with_formulas.csv")
        self.q_table_ = q_table
        self.feature_selection_time_ = time.perf_counter() - fs_start
        print(f"[Timing] Total feature selection time: {self.feature_selection_time_:.4f} seconds")
    
        return self

    def get_selected_features(self, feature_names: List[str]) -> List[str]:
        if self.best_mask_ is None:
            raise ValueError("Model chưa fit.")
        return [f for f, m in zip(feature_names, self.best_mask_) if m == 1]

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        if self.best_mask_ is None:
            raise ValueError("Model chưa fit.")
        selected_cols = X.columns[self.best_mask_ == 1]
        return X[selected_cols]

    def fit_transform(self, X: pd.DataFrame, y: np.ndarray) -> pd.DataFrame:
        self.fit(X, y)
        return self.transform(X)

In [30]:
if __name__ == "__main__":
    import pandas as pd

    # 1) Đọc dữ liệu local/Colab trực tiếp.
    X_path = resolve_data_path("X_train.csv")
    y_path = resolve_data_path("y_train.csv")

    X, y = load_data(X_path, y_path)

    # 2) Config
    config = QLDEConfig(
        population_size=20,
        max_generations=100,
        crossover_rate=0.9,
        init_F_min=0.3,
        init_F_max=0.8,
        q_learning_rate=0.1,
        q_discount=0.9,
        softmax_temperature=1.0,
        alpha_metric=0.95,
        beta_size=0.05,
        actions=(-0.05, 0.0, 0.05),
        threshold=0.5,
        min_features=1,
        random_state=42,

        # ===== thêm cho diversity archive =====
        archive_size=15,
        archive_min_hamming=0.10,
        archive_novelty_k=3,
        leader_novelty_weight=0.7,
        leader_fitness_weight=0.3
    )

    # 3) Tạo selector và fit tuần tự/local
    selector = QLDEFeatureSelector(config=config)
    selector.fit(X, y)

    print("Total FS time:", selector.feature_selection_time_)

    selected_features = selector.get_selected_features(list(X.columns))
    X_selected = selector.transform(X)

    print("\n========== KẾT QUẢ CUỐI ==========")
    print(f"Best fitness       : {selector.best_fitness_:.6f}")
    print(f"Best macro-F1      : {selector.best_macro_f1_:.6f}")
    print(f"Số feature đã chọn : {selector.best_num_features_}")
    print(f"Tên feature chọn   : {selected_features}")

    print("\nQ-table cuối:")
    print(selector.q_table_)

    # 4) Lưu file kết quả
    X_selected.to_csv(output_path("X_train_selected.csv"), index=False)
    pd.DataFrame({"selected_feature": selected_features}).to_csv(
        output_path("selected_features_list.csv"), index=False
    )
    pd.DataFrame(selector.history_).to_csv(output_path("qlde_history.csv"), index=False)

    pd.DataFrame(selector.population_fitness_history_).to_csv(
        output_path("population_fitness_history.csv"), index=False
    )

    pd.DataFrame(selector.population_state_history_).to_csv(
        output_path("population_state_history.csv"), index=False
    )

    print("\nĐã lưu:")
    print("- X_train_selected.csv")
    print("- selected_features_list.csv")
    print("- qlde_history.csv")


[Timing] Initial population evaluation: 24.6458 seconds


Generation 01 | Best fitness = 0.68206 | Best macro-F1 = 0.74539 | Selected features = 517 | Improvements = 9 | Archive = 15 | Eval time = 24.2331s | Total gen time = 24.2337s


Generation 02 | Best fitness = 0.68206 | Best macro-F1 = 0.74539 | Selected features = 517 | Improvements = 8 | Archive = 15 | Eval time = 23.8452s | Total gen time = 23.8457s


Generation 03 | Best fitness = 0.68968 | Best macro-F1 = 0.75403 | Selected features = 533 | Improvements = 7 | Archive = 15 | Eval time = 22.8861s | Total gen time = 22.8865s


Generation 04 | Best fitness = 0.68968 | Best macro-F1 = 0.75403 | Selected features = 533 | Improvements = 7 | Archive = 15 | Eval time = 23.9157s | Total gen time = 23.9160s


Generation 05 | Best fitness = 0.68968 | Best macro-F1 = 0.75403 | Selected features = 533 | Improvements = 10 | Archive = 15 | Eval time = 23.7760s | Total gen time = 23.7765s


... (91 outputs left out)

Generation 97 | Best fitness = 0.73042 | Best macro-F1 = 0.79560 | Selected features = 508 | Improvements = 0 | Archive = 15 | Eval time = 21.1106s | Total gen time = 21.1110s


Generation 98 | Best fitness = 0.73042 | Best macro-F1 = 0.79560 | Selected features = 508 | Improvements = 0 | Archive = 15 | Eval time = 20.7889s | Total gen time = 20.7893s


Generation 99 | Best fitness = 0.73128 | Best macro-F1 = 0.79793 | Selected features = 535 | Improvements = 2 | Archive = 15 | Eval time = 20.7315s | Total gen time = 20.7318s


Generation 100 | Best fitness = 0.73128 | Best macro-F1 = 0.79793 | Selected features = 535 | Improvements = 1 | Archive = 15 | Eval time = 20.7842s | Total gen time = 20.7845s
[Timing] Total feature selection time: 2188.0293 seconds
Total FS time: 2188.029267497999

========== KẾT QUẢ CUỐI ==========
Best fitness       : 0.731285
Best macro-F1      : 0.797931
Số feature đã chọn : 535
Tên feature chọn   : ['AAK1', 'ABCA1', 'ACAA1', 'ACAP1', 'ACP5', 'ACTN4', 'ADAM15', 'ADAR', 'ADH5', 'AGR2', 'AGTRAP', 'AKAP13', 'ALCAM', 'ALDH1A1', 'ALDH2', 'ALOX5AP', 'ANAPC11', 'ANXA1', 'ANXA5', 'AP1S1', 'AP3S1', 'APLP2', 'APOE', 'APP', 'ARF3', 'ARF6', 'ARHGAP18', 'ARL2', 'ARL4A', 'ARPC1A', 'ASAH1', 'ASNA1', 'ATF3', 'ATOX1', 'ATP11A', 'ATP1B1', 'ATP1B3', 'ATP2A2', 'ATP6AP1', 'ATP6V0D1', 'B4GALT1', 'B4GALT5', 'BAG1', 'BHLHE40', 'BLVRA', 'BLVRB', 'BNIP3', 'BNIP3L', 'BRI3', 'BSG', 'BST2', 'BTG1', 'BTG2', 'BUD31', 'C10orf11', 'C15orf48', 'C16orf89', 'C17orf89', 'C19orf33', 'C19orf60', 'C1QA', 'C1QB', 'C1QC'

In [31]:
import pandas as pd

# Đọc dữ liệu
X_train = pd.read_csv(resolve_data_path("X_train.csv"))
X_val = pd.read_csv(resolve_data_path("X_val.csv"))
X_test = pd.read_csv(resolve_data_path("X_test.csv"))

y_train = pd.read_csv(resolve_data_path("y_train.csv")).iloc[:, 0].values
y_val = pd.read_csv(resolve_data_path("y_val.csv")).iloc[:, 0].values
y_test = pd.read_csv(resolve_data_path("y_test.csv")).iloc[:, 0].values

# Áp feature selected lên cả 3 tập
X_train_selected = X_train[selected_features].copy()
X_val_selected = X_val[selected_features].copy()
X_test_selected = X_test[selected_features].copy()

print("Train selected shape:", X_train_selected.shape)
print("Val selected shape:", X_val_selected.shape)
print("Test selected shape:", X_test_selected.shape)

Train selected shape: (16000, 535)
Val selected shape: (2000, 535)
Test selected shape: (2000, 535)


In [32]:
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score

def compute_metrics(y_true, y_pred, model_name):
    report = classification_report(y_true, y_pred,
                                   target_names=CLASS_NAMES,
                                   output_dict=True, zero_division=0)
    row = {"model": model_name}
    for cls in CLASS_NAMES:
        for m in ["precision","recall","f1-score"]:
            col = f"{cls}_{m.replace('-score','')}"
            row[col] = round(report[cls][m], 4)
    for avg in ["macro avg","weighted avg"]:
        prefix = avg.replace(" avg","")
        for m in ["precision","recall","f1-score"]:
            col = f"{prefix}_{m.replace('-score','')}"
            row[col] = round(report[avg][m], 4)
    row["accuracy"] = round(report["accuracy"], 4)
    return row

In [33]:
all_results = [] 

from sklearn.neighbors         import KNeighborsClassifier
from sklearn.tree              import DecisionTreeClassifier
from sklearn.ensemble          import RandomForestClassifier
from sklearn.naive_bayes       import GaussianNB
from sklearn.linear_model      import LogisticRegression
from sklearn.svm               import LinearSVC, SVC
from sklearn.model_selection   import GridSearchCV
from sklearn.pipeline          import Pipeline
import numpy as np

unique_classes = np.unique(y_train)
N_CLASSES = len(unique_classes)
CLASS_NAMES = [str(c) for c in unique_classes]

print("N_CLASSES :", N_CLASSES)
print("CLASS_NAMES:", CLASS_NAMES)

ML_CONFIGS = [
    (
        "KNN",
        KNeighborsClassifier(),
        {"n_neighbors":[3,5,7,11], "weights":["uniform","distance"],
         "metric":["euclidean","manhattan"]}
    ),
    (
        "DecisionTree",
        DecisionTreeClassifier(random_state=42),
        {"max_depth":[None,10,20,30], "min_samples_split":[2,5,10],
         "criterion":["gini","entropy"]}
    ),
    (
        "RandomForest",
        RandomForestClassifier(random_state=42, n_jobs=-1),
        {"n_estimators":[100,200], "max_depth":[None,10,20],
         "min_samples_split":[2,5]}
    ),
    (
        "GaussianNB",
        GaussianNB(),
        {"var_smoothing":[1e-9,1e-8,1e-7,1e-6]}
    ),
    (
        "LogisticRegression",
        LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1),
        {"C":[0.01,0.1,1,10], "solver":["lbfgs","saga"],
         "multi_class":["multinomial"]}
    ),
    (
        "LinearSVM",
        LinearSVC(max_iter=2000, random_state=42),
        {"C":[0.01,0.1,1,10]}
    ),
    (
        "RBF_SVM",
        SVC(kernel="rbf", random_state=42),
        {"C":[0.1,1,10], "gamma":["scale","auto"]}
    ),
]

import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

all_results = []
report_rows = []

for name, estimator, param_grid in ML_CONFIGS:
    print(f"\n[{name}] GridSearch đang chạy...")

    gs = GridSearchCV(
        estimator,
        param_grid,
        cv=5,
        scoring="f1_macro",
        n_jobs=-1,
        verbose=0
    )

    gs.fit(X_train_selected, y_train)

    best = gs.best_estimator_
    y_pred = best.predict(X_test_selected)

    print(f"  Best params : {gs.best_params_}")
    print(f"  CV F1-macro : {gs.best_score_:.4f}")

    # report dạng dict thay vì in text thô
    report_dict = classification_report(
        y_test,
        y_pred,
        target_names=CLASS_NAMES,
        zero_division=0,
        output_dict=True
    )

    # Gom toàn bộ report của model thành 1 hàng
    flat_row = {"model": name}

    for label, metrics in report_dict.items():
        if isinstance(metrics, dict):  # các class, macro avg, weighted avg
            for metric_name, value in metrics.items():
                flat_row[f"{label}_{metric_name}"] = value
        else:  # accuracy là scalar
            flat_row[label] = metrics

    flat_row["best_params"] = str(gs.best_params_)
    flat_row["cv_f1_macro"] = gs.best_score_
    report_rows.append(flat_row)

    # phần cũ của bạn nếu vẫn muốn giữ
    row = compute_metrics(y_test, y_pred, name)
    row["best_params"] = str(gs.best_params_)
    all_results.append(row)

# Bảng classification report: mỗi hàng là 1 model
classification_report_df = pd.DataFrame(report_rows)

# Sắp xếp cột cho dễ nhìn
first_cols = ["model", "accuracy", "cv_f1_macro", "best_params"]
other_cols = [c for c in classification_report_df.columns if c not in first_cols]
classification_report_df = classification_report_df[first_cols + other_cols]

display(classification_report_df)

pd.DataFrame(all_results).to_csv(
    output_path("all_results_models.csv"),
    index=False,
    encoding="utf-8-sig"
)

N_CLASSES : 8
CLASS_NAMES: ['0', '1', '2', '3', '4', '5', '6', '7']

[KNN] GridSearch đang chạy...
  Best params : {'metric': 'manhattan', 'n_neighbors': 11, 'weights': 'distance'}
  CV F1-macro : 0.5566

[DecisionTree] GridSearch đang chạy...
  Best params : {'criterion': 'gini', 'max_depth': 10, 'min_samples_split': 5}
  CV F1-macro : 0.6305

[RandomForest] GridSearch đang chạy...
  Best params : {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 200}
  CV F1-macro : 0.7837

[GaussianNB] GridSearch đang chạy...
  Best params : {'var_smoothing': 1e-06}
  CV F1-macro : 0.3841

[LogisticRegression] GridSearch đang chạy...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and wi

  Best params : {'C': 0.01, 'multi_class': 'multinomial', 'solver': 'saga'}
  CV F1-macro : 0.7843

[LinearSVM] GridSearch đang chạy...
  Best params : {'C': 0.01}
  CV F1-macro : 0.7824

[RBF_SVM] GridSearch đang chạy...
  Best params : {'C': 10, 'gamma': 'auto'}
  CV F1-macro : 0.8448


,model,accuracy,cv_f1_macro,best_params,0_precision,0_recall,0_f1-score,0_support,1_precision,1_recall,...,7_f1-score,7_support,macro avg_precision,macro avg_recall,macro avg_f1-score,macro avg_support,weighted avg_precision,weighted avg_recall,weighted avg_f1-score,weighted avg_support
0,KNN,0.5900,0.556638,"{'metric': 'manhattan', 'n_neighbors': 11, 'we...",0.643678,0.224,0.332344,250.0,0.391379,0.908,...,0.699074,250.0,0.647264,0.5900,0.583872,2000.0,0.647264,0.5900,0.583872,2000.0
1,DecisionTree,0.6485,0.630478,"{'criterion': 'gini', 'max_depth': 10, 'min_sa...",0.519417,0.428,0.469298,250.0,0.690391,0.776,...,0.661626,250.0,0.647308,0.6485,0.646274,2000.0,0.647308,0.6485,0.646274,2000.0
2,RandomForest,0.7910,0.783652,"{'max_depth': None, 'min_samples_split': 5, 'n...",0.791908,0.548,0.647754,250.0,0.765517,0.888,...,0.815451,250.0,0.797926,0.7910,0.788923,2000.0,0.797926,0.7910,0.788923,2000.0
3,GaussianNB,0.3970,0.384054,{'var_smoothing': 1e-06},0.252874,0.088,0.130564,250.0,0.382562,0.860,...,0.527132,250.0,0.438178,0.3970,0.382931,2000.0,0.438178,0.3970,0.382931,2000.0
4,LogisticRegression,0.7905,0.784263,"{'C': 0.01, 'multi_class': 'multinomial', 'sol...",0.717593,0.620,0.665236,250.0,0.829630,0.896,...,0.800821,250.0,0.788404,0.7905,0.788660,2000.0,0.788404,0.7905,0.788660,2000.0
5,LinearSVM,0.7960,0.782371,{'C': 0.01},0.769634,0.588,0.666667,250.0,0.833935,0.924,...,0.804124,250.0,0.794257,0.7960,0.792951,2000.0,0.794257,0.7960,0.792951,2000.0
6,RBF_SVM,0.8495,0.844826,"{'C': 10, 'gamma': 'auto'}",0.771930,0.704,0.736402,250.0,0.882812,0.904,...,0.865306,250.0,0.849723,0.8495,0.848934,2000.0,0.849723,0.8495,0.848934,2000.0
